# Bài 4: Bài Tập Thực Hành - Cross-Validation và Tuning

**Hướng dẫn:**
- Hoàn thành các câu hỏi và bài tập dưới đây để củng cố kiến thức về kiểm định chéo và tìm kiếm siêu tham số.
- Cố gắng tự trả lời trước khi xem đáp án.
- Chạy các ô code để kiểm tra kết quả của bạn.

---

### Phần 1: Câu hỏi lý thuyết

**Câu 1:** Tại sao K-Fold Cross-Validation lại cho một ước tính hiệu suất mô hình đáng tin cậy hơn so với một lần chia train-test duy nhất?

**Câu 2:** Khi sử dụng `lgb.cv` hoặc các phương pháp cross-validation khác cho bài toán phân loại (classification), tại sao việc đặt `stratified=True` lại quan trọng, đặc biệt là với dữ liệu mất cân bằng?

**Câu 3:** So sánh Grid Search và Random Search. Trong tình huống nào bạn sẽ ưu tiên sử dụng Random Search hơn Grid Search?

**Câu 4:** Bayesian Optimization được cho là thông minh hơn Random Search. Giải thích ngắn gọn ý tưởng đằng sau Bayesian Optimization và tại sao nó có thể hiệu quả hơn.

**Câu 5:** Khi sử dụng `RandomizedSearchCV` của Scikit-learn, tham số `n_iter` và `cv` có ý nghĩa gì? Tăng các giá trị này ảnh hưởng đến quá trình tìm kiếm như thế nào?

---

### Phần 2: Bài tập thực hành

**Bối cảnh:** Chúng ta sẽ sử dụng bộ dữ liệu dự đoán bệnh tiểu đường Pima Indians (`diabetes.csv`). Mục tiêu là xây dựng và tinh chỉnh một mô hình LightGBM để dự đoán một người có bị bệnh tiểu đường hay không (`Outcome`).

In [ ]:
import pandas as pd
import lightgbm as lgb

# Tải dữ liệu
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
col_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df_diabetes = pd.read_csv(url, header=None, names=col_names)

X = df_diabetes.drop('Outcome', axis=1)
y = df_diabetes['Outcome']

df_diabetes.head()

**Bài tập 6: Đánh giá mô hình cơ bản với `lgb.cv`**

1.  Tạo một đối tượng `lgb.Dataset` từ `X` và `y`.
2.  Định nghĩa một bộ tham số cơ bản (ví dụ: `objective='binary'`, `metric='auc'`, `random_state=42`).
3.  Sử dụng `lgb.cv` để thực hiện kiểm định chéo 5-fold (`nfold=5`).
4.  In ra điểm AUC trung bình tốt nhất và số cây tối ưu.

In [ ]:
# Viết code của bạn vào đây
lgb_data = ...

params_base = {
    'objective': 'binary',
    'metric': 'auc',
    'random_state': 42,
    'verbose': -1
}

cv_results = lgb.cv(
    params=params_base,
    train_set=lgb_data,
    nfold=5,
    stratified=True,
    shuffle=True,
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50, verbose=False)],
    seed=42
)

print("--- Đánh giá với lgb.cv ---")
print(f"Điểm AUC trung bình tốt nhất: {cv_results['valid auc-mean'][-1]:.4f}")
print(f"Số cây tối ưu: {len(cv_results['valid auc-mean'])}")

**Bài tập 7: Tinh chỉnh tham số với `GridSearchCV`**

1.  Định nghĩa một lưới tìm kiếm (`param_grid`) nhỏ cho `GridSearchCV`. Ví dụ:
    - `learning_rate`: [0.01, 0.05, 0.1]
    - `num_leaves`: [10, 20, 30]
    - `n_estimators`: [100, 200]
2.  Khởi tạo một mô hình `lgb.LGBMClassifier`.
3.  Khởi tạo `GridSearchCV` với mô hình, lưới tham số, `cv=3`, và `scoring='roc_auc'`.
4.  Chạy tìm kiếm trên toàn bộ dữ liệu (`X`, `y`).
5.  In ra bộ tham số tốt nhất và điểm số tương ứng.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Viết code của bạn vào đây
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [10, 20, 30],
    'n_estimators': [100, 200]
}

lgbm = lgb.LGBMClassifier(objective='binary', random_state=42)

grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print("\n--- Kết quả GridSearchCV ---")
print(f"Tham số tốt nhất: {grid_search.best_params_}")
print(f"Điểm AUC tốt nhất: {grid_search.best_score_:.4f}")

**Câu hỏi tư duy:** So sánh kết quả AUC từ `lgb.cv` và `GridSearchCV`. Tại sao chúng có thể khác nhau? `GridSearchCV` đã thử bao nhiêu mô hình tất cả?

---

## Đáp Án

### Phần 1: Lý thuyết

**Câu 1:**
K-Fold Cross-Validation đáng tin cậy hơn vì nó đánh giá mô hình trên K tập con khác nhau của dữ liệu. Mỗi điểm dữ liệu đều được dùng làm dữ liệu kiểm tra đúng một lần. Bằng cách lấy trung bình kết quả từ K lần đánh giá, nó làm giảm yếu tố may rủi do một lần chia cụ thể gây ra, từ đó cho một ước tính ổn định và khách quan hơn về khả năng tổng quát hóa của mô hình trên dữ liệu mới.

**Câu 2:**
`stratified=True` đảm bảo rằng tỷ lệ của các lớp trong mỗi fold là tương tự như tỷ lệ của chúng trong toàn bộ tập dữ liệu. Điều này cực kỳ quan trọng với dữ liệu mất cân bằng. Nếu không có stratified, một fold có thể vô tình chứa rất ít hoặc không có mẫu nào của lớp thiểu số, làm cho việc đánh giá trên fold đó trở nên vô nghĩa và kết quả cross-validation bị sai lệch.

**Câu 3:**
- **Grid Search:** Thử tất cả các kết hợp tham số trong một lưới được định nghĩa trước. Đảm bảo tìm ra điểm tối ưu trong lưới đó nhưng rất tốn kém.
- **Random Search:** Thử một số lượng ngẫu nhiên các kết hợp tham số từ một không gian (hoặc phân phối) được định nghĩa trước. Hiệu quả hơn nhiều.
Bạn nên ưu tiên Random Search khi:
1.  Không gian tìm kiếm lớn (nhiều tham số hoặc nhiều giá trị cho mỗi tham số).
2.  Tài nguyên tính toán có hạn.
3.  Bạn nghi ngờ rằng chỉ một vài tham số thực sự quan trọng. Random Search có cơ hội khám phá các giá trị đa dạng hơn cho các tham số quan trọng đó, trong khi Grid Search có thể lãng phí thời gian vào các tham số không ảnh hưởng nhiều.

**Câu 4:**
Bayesian Optimization xây dựng một mô hình xác suất (gọi là surrogate model) để ước tính hàm mục tiêu (ví dụ: điểm AUC) trông như thế nào. Sau mỗi lần thử một bộ tham số và nhận kết quả, nó cập nhật mô hình xác suất này. Dựa trên mô hình đã học, nó sẽ chọn điểm tiếp theo để thử nghiệm một cách thông minh: cân bằng giữa việc **khai thác (exploitation)** các vùng hứa hẹn (nơi có điểm số cao) và **khám phá (exploration)** các vùng chưa chắc chắn. Điều này giúp nó hội tụ đến điểm tối ưu nhanh hơn so với việc tìm kiếm ngẫu nhiên hoặc mù quáng của Random Search.

**Câu 5:**
- **`n_iter` (number of iterations):** Là số lượng bộ tham số ngẫu nhiên sẽ được chọn và đánh giá. Đây là tham số chính để kiểm soát ngân sách tính toán của Random Search. Tăng `n_iter` sẽ làm quá trình tìm kiếm kỹ hơn nhưng tốn nhiều thời gian hơn.
- **`cv` (cross-validation):** Là số lượng fold sẽ được sử dụng để đánh giá mỗi bộ tham số. Ví dụ, `cv=3` có nghĩa là mỗi bộ tham số sẽ được huấn luyện và đánh giá 3 lần. Tăng `cv` sẽ làm cho việc đánh giá mỗi bộ tham số trở nên đáng tin cậy hơn, nhưng cũng làm tăng tổng thời gian tìm kiếm lên `cv` lần.

### Phần 2: Thực hành (Code đáp án)

In [ ]:
# Bài tập 6: Đánh giá mô hình cơ bản với lgb.cv
lgb_data = lgb.Dataset(X, label=y)

params_base = {
    'objective': 'binary',
    'metric': 'auc',
    'random_state': 42,
    'verbose': -1
}

cv_results = lgb.cv(
    params=params_base,
    train_set=lgb_data,
    nfold=5,
    stratified=True,
    shuffle=True,
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50, verbose=False)],
    seed=42
)

print("--- Đánh giá với lgb.cv ---")
print(f"Điểm AUC trung bình tốt nhất: {cv_results['valid auc-mean'][-1]:.4f}")
print(f"Số cây tối ưu: {len(cv_results['valid auc-mean'])}")

In [ ]:
# Bài tập 7: Tinh chỉnh tham số với GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [10, 20, 30],
    'n_estimators': [100, 200]
}

lgbm = lgb.LGBMClassifier(objective='binary', random_state=42)

grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print("\n--- Kết quả GridSearchCV ---")
print(f"Tham số tốt nhất: {grid_search.best_params_}")
print(f"Điểm AUC tốt nhất: {grid_search.best_score_:.4f}")

**Câu trả lời cho câu hỏi tư duy:**
Kết quả AUC có thể khác nhau vì:
1.  **Bộ tham số khác nhau:** `lgb.cv` chạy với một bộ tham số cơ bản duy nhất, trong khi `GridSearchCV` đã thử nhiều bộ tham số và trả về kết quả của bộ tốt nhất mà nó tìm thấy.
2.  **Số folds khác nhau:** `lgb.cv` dùng 5 folds, trong khi `GridSearchCV` dùng 3 folds. Số lượng folds khác nhau sẽ dẫn đến các cách chia dữ liệu khác nhau và kết quả đánh giá cũng khác nhau.
3.  **Early Stopping:** `lgb.cv` sử dụng early stopping để tìm số cây tối ưu, trong khi `GridSearchCV` trong ví dụ này đã chạy với số cây cố định (`n_estimators` = 100 hoặc 200).

`GridSearchCV` đã thử tổng cộng: 3 (`learning_rate`) * 3 (`num_leaves`) * 2 (`n_estimators`) = **18 bộ tham số**. Với `cv=3`, tổng số mô hình đã được huấn luyện là 18 * 3 = **54 mô hình**.